# Load FEMA Public Assistance Funded Projects into Delta

Pulls the OpenFEMA `PublicAssistanceFundedProjectsDetails` (v2) dataset,
filtered to recent years so the demo table stays fast, and writes it to
`workspace.default.funded_projects_details`.

No API key needed. OpenFEMA is free and read-only.

Two paths below:
- **Cell A (recommended):** robust paging loop. Always works.
- **Cell B (faster, BETA):** one-shot parquet download via `$allrecords=true`.

Run A *or* B, then the write cell. Tighten the date filter if the table is bigger than you want.

In [0]:
# Config. Change these two to control size.
START_DATE = "2021-01-01"     # only declarations on/after this date
TABLE_NAME = "workspace.default.funded_projects_details"

ENDPOINT = "https://www.fema.gov/api/open/v2/PublicAssistanceFundedProjectsDetails"
FILTER = f"declarationDate ge '{START_DATE}'"

## Cell A — robust paging loop (recommended)
Pages 10,000 rows at a time using `$skip`, stops when a short page comes back.

In [0]:
import requests
import pandas as pd

PAGE = 10000
params_base = {
    "$filter": FILTER,
    "$format": "jsona",   # pure array, no metadata wrapper
    "$orderby": "declarationDate",
    "$top": str(PAGE),
}

rows = []
skip = 0
while True:
    params = dict(params_base, **{"$skip": str(skip)})
    r = requests.get(ENDPOINT, params=params, timeout=120)
    r.raise_for_status()
    batch = r.json()            # jsona returns a bare list of record dicts
    if not batch:
        break
    rows.extend(batch)
    print(f"pulled {len(batch):>6} rows (running total {len(rows)})")
    if len(batch) < PAGE:
        break
    skip += PAGE

pdf = pd.DataFrame(rows)
print(f"\nTotal rows: {len(pdf):,}   columns: {len(pdf.columns)}")

pulled  10000 rows (running total 10000)
pulled  10000 rows (running total 20000)
pulled  10000 rows (running total 30000)
pulled  10000 rows (running total 40000)
pulled  10000 rows (running total 50000)
pulled  10000 rows (running total 60000)
pulled  10000 rows (running total 70000)
pulled   7883 rows (running total 77883)

Total rows: 77,883   columns: 25


## Cell B — one-shot parquet download (faster, BETA)
Skip Cell A and run this instead if you want the whole filtered set in one call.
`$allrecords=true` is a BETA feature and must be paired with a file download.

In [0]:
# import requests, pandas as pd
# url = (
#     f"{ENDPOINT}?$filter={FILTER}"
#     "&$format=parquet&$allrecords=true&$filename=pa.parquet"
# )
# local = "/tmp/pa.parquet"
# with requests.get(url, stream=True, timeout=600) as r:
#     r.raise_for_status()
#     with open(local, "wb") as f:
#         for chunk in r.iter_content(chunk_size=1 << 20):
#             f.write(chunk)
# pdf = pd.read_parquet(local)
# print(f"Total rows: {len(pdf):,}   columns: {len(pdf.columns)}")

## Write to Delta
Casts the money and date columns so Genie and the SQL functions behave.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

sdf = spark.createDataFrame(pdf)

# Type the columns the tools rely on. API delivers most as strings.
money_cols = [
    "projectAmount", "federalShareObligated", "totalObligated", "mitigationAmount",
]
for c in money_cols:
    if c in sdf.columns:
        sdf = sdf.withColumn(c, F.col(c).cast(DecimalType(18, 2)))

if "disasterNumber" in sdf.columns:
    sdf = sdf.withColumn("disasterNumber", F.col("disasterNumber").cast("int"))
if "declarationDate" in sdf.columns:
    sdf = sdf.withColumn("declarationDate", F.to_date("declarationDate"))

(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Wrote {sdf.count():,} rows to {TABLE_NAME}")

Wrote 77,883 rows to workspace.default.funded_projects_details


In [0]:
# Quick sanity check: top states by federal share obligated
display(
    spark.sql(f"""
        SELECT stateAbbreviation,
               COUNT(*)                   AS projects,
               SUM(federalShareObligated) AS total_federal_share
        FROM {TABLE_NAME}
        GROUP BY stateAbbreviation
        ORDER BY total_federal_share DESC
        LIMIT 10
    """)
)

stateAbbreviation,projects,total_federal_share
FL,10125,6214063049.59
LA,5113,4003323457.38
NC,3787,2715524759.98
PR,3307,1998585593.69
GA,2497,1834200666.48
TX,3716,1300521413.20
KY,7644,1172378309.35
NY,3249,1171083311.32
CA,4763,1170304616.47
TN,2572,1074203926.92


In [0]:
from datetime import date
from decimal import Decimal
from enum import Enum
from typing import Optional

from pydantic import BaseModel, Field, field_validator

_TABLE = "workspace.default.funded_projects_details"


# --- Shared enums / value objects -------------------------------------------

class ProjectSize(str, Enum):
    large = "Large"
    small = "Small"


class USState(BaseModel):
    """Two-letter state or territory code, validated and upper-cased."""
    code: str = Field(..., description="Two-letter USPS state/territory code, e.g. 'TX'.")

    @field_validator("code")
    @classmethod
    def _upper_two(cls, v: str) -> str:
        v = v.strip().upper()
        if len(v) != 2 or not v.isalpha():
            raise ValueError("state code must be two letters, e.g. 'FL'")
        return v


# --- Tool 1: funding totals by jurisdiction ---------------------------------

class FundingByJurisdictionInput(BaseModel):
    """Inputs for summarizing obligated PA funding for a state or county."""
    state: USState = Field(..., description="State or territory to summarize.")
    county: Optional[str] = Field(
        None,
        description="Optional county/parish name to narrow the result. "
                    "Omit for a statewide total.",
    )
    start_date: Optional[date] = Field(
        None, description="Only include declarations on or after this date."
    )
    end_date: Optional[date] = Field(
        None, description="Only include declarations on or before this date."
    )
    incident_type: Optional[str] = Field(
        None, description="Optional incident type filter, e.g. 'Hurricane', 'Flood'."
    )

    @field_validator("end_date")
    @classmethod
    def _check_order(cls, v, info):
        start = info.data.get("start_date")
        if v and start and v < start:
            raise ValueError("end_date cannot be before start_date")
        return v


class FundingByJurisdictionResult(BaseModel):
    state: str
    county: Optional[str]
    project_count: int
    total_federal_share_obligated: Decimal
    total_obligated: Decimal


def summarize_pa_funding_by_jurisdiction(
    args: FundingByJurisdictionInput,
) -> FundingByJurisdictionResult:
    """
    Agent goal: "How much PA funding did <place> receive?"

    Suggested Unity Catalog SQL function body:

        SELECT
            stateAbbreviation,
            :county AS county,
            COUNT(*)                      AS project_count,
            SUM(federalShareObligated)    AS total_federal_share_obligated,
            SUM(totalObligated)           AS total_obligated
        FROM workspace.default.funded_projects_details
        WHERE stateAbbreviation = :state_code
          AND (:county IS NULL OR county = :county)
          AND (:start_date IS NULL OR declarationDate >= :start_date)
          AND (:end_date   IS NULL OR declarationDate <= :end_date)
          AND (:incident_type IS NULL OR incidentType = :incident_type)
        GROUP BY stateAbbreviation
    """
    params = {
        "state_code": args.state.code,
        "county": args.county,
        "start_date": str(args.start_date) if args.start_date else None,
        "end_date": str(args.end_date) if args.end_date else None,
        "incident_type": args.incident_type,
    }
    row = spark.sql(
        f"""
        SELECT
            stateAbbreviation,
            COUNT(*)                   AS project_count,
            SUM(federalShareObligated) AS total_federal_share_obligated,
            SUM(totalObligated)        AS total_obligated
        FROM {_TABLE}
        WHERE stateAbbreviation = :state_code
          AND (:county IS NULL OR county = :county)
          AND (:start_date IS NULL OR declarationDate >= :start_date)
          AND (:end_date   IS NULL OR declarationDate <= :end_date)
          AND (:incident_type IS NULL OR incidentType = :incident_type)
        GROUP BY stateAbbreviation
        """,
        args=params,
    ).first()

    if row is None:
        return FundingByJurisdictionResult(
            state=args.state.code,
            county=args.county,
            project_count=0,
            total_federal_share_obligated=Decimal("0"),
            total_obligated=Decimal("0"),
        )
    return FundingByJurisdictionResult(
        state=row.stateAbbreviation,
        county=args.county,
        project_count=row.project_count,
        total_federal_share_obligated=row.total_federal_share_obligated or Decimal("0"),
        total_obligated=row.total_obligated or Decimal("0"),
    )


# --- Tool 2: funding breakdown by damage category ---------------------------

class FundingByDamageCategoryInput(BaseModel):
    """Inputs for breaking down a single disaster's funding by work category."""
    disaster_number: int = Field(
        ..., ge=1, description="FEMA disaster number to break down."
    )
    emergency_work_only: bool = Field(
        False,
        description="If true, limit to emergency-work categories A and B "
                    "(debris removal, protective measures).",
    )


class DamageCategoryRow(BaseModel):
    damage_category_code: str
    damage_category_description: str
    project_count: int
    total_federal_share_obligated: Decimal


class FundingByDamageCategoryResult(BaseModel):
    disaster_number: int
    rows: list[DamageCategoryRow]


def breakdown_disaster_funding_by_category(
    args: FundingByDamageCategoryInput,
) -> FundingByDamageCategoryResult:
    """
    Agent goal: "Break down disaster <N> funding by category of work."

    Suggested Unity Catalog SQL function body:

        SELECT
            damageCategoryCode,
            MAX(damageCategoryDescrip)   AS damage_category_description,
            COUNT(*)                     AS project_count,
            SUM(federalShareObligated)   AS total_federal_share_obligated
        FROM workspace.default.funded_projects_details
        WHERE disasterNumber = :disaster_number
          AND (:emergency_work_only = false
               OR damageCategoryCode IN ('A','B'))
        GROUP BY damageCategoryCode
        ORDER BY total_federal_share_obligated DESC
    """
    params = {
        "disaster_number": args.disaster_number,
        "emergency_work_only": args.emergency_work_only,
    }
    rows = spark.sql(
        f"""
        SELECT
            damageCategoryCode,
            MAX(damageCategoryDescrip)  AS damage_category_description,
            COUNT(*)                    AS project_count,
            SUM(federalShareObligated)  AS total_federal_share_obligated
        FROM {_TABLE}
        WHERE disasterNumber = :disaster_number
          AND (:emergency_work_only = FALSE
               OR damageCategoryCode IN ('A', 'B'))
        GROUP BY damageCategoryCode
        ORDER BY total_federal_share_obligated DESC
        """,
        args=params,
    ).collect()

    return FundingByDamageCategoryResult(
        disaster_number=args.disaster_number,
        rows=[
            DamageCategoryRow(
                damage_category_code=r.damageCategoryCode,
                damage_category_description=r.damage_category_description or "",
                project_count=r.project_count,
                total_federal_share_obligated=r.total_federal_share_obligated or Decimal("0"),
            )
            for r in rows
        ],
    )


# --- Tool 3: largest projects (oversight lens) ------------------------------

class LargestProjectsInput(BaseModel):
    """Inputs for ranking the biggest obligated projects."""
    limit: int = Field(
        10, ge=1, le=100, description="How many top projects to return (1-100)."
    )
    state: Optional[USState] = Field(
        None, description="Optional state filter."
    )
    disaster_number: Optional[int] = Field(
        None, ge=1, description="Optional disaster number filter."
    )
    project_size: Optional[ProjectSize] = Field(
        None, description="Optional filter: Large or Small."
    )


class ProjectRow(BaseModel):
    pw_number: int
    application_title: str
    state: str
    county: Optional[str]
    damage_category_description: str
    federal_share_obligated: Decimal


class LargestProjectsResult(BaseModel):
    rows: list[ProjectRow]


def rank_largest_pa_projects(args: LargestProjectsInput) -> LargestProjectsResult:
    """
    Agent goal: "What were the largest PA projects (where did the money go)?"

    Suggested Unity Catalog SQL function body:

        SELECT
            pwNumber,
            applicationTitle,
            stateAbbreviation,
            county,
            damageCategoryDescrip,
            federalShareObligated
        FROM workspace.default.funded_projects_details
        WHERE (:state_code     IS NULL OR stateAbbreviation = :state_code)
          AND (:disaster_number IS NULL OR disasterNumber = :disaster_number)
          AND (:project_size    IS NULL OR projectSize = :project_size)
        ORDER BY federalShareObligated DESC
        LIMIT :limit
    """
    state_code = args.state.code if args.state else None
    params = {
        "state_code": state_code,
        "disaster_number": args.disaster_number,
        "project_size": args.project_size.value if args.project_size else None,
        "limit": args.limit,
    }
    rows = spark.sql(
        f"""
        SELECT
            pwNumber,
            applicationTitle,
            stateAbbreviation,
            county,
            damageCategoryDescrip,
            federalShareObligated
        FROM {_TABLE}
        WHERE (:state_code     IS NULL OR stateAbbreviation = :state_code)
          AND (:disaster_number IS NULL OR disasterNumber    = :disaster_number)
          AND (:project_size    IS NULL OR projectSize       = :project_size)
        ORDER BY federalShareObligated DESC
        LIMIT :limit
        """,
        args=params,
    ).collect()

    return LargestProjectsResult(
        rows=[
            ProjectRow(
                pw_number=r.pwNumber,
                application_title=r.applicationTitle or "",
                state=r.stateAbbreviation,
                county=r.county,
                damage_category_description=r.damageCategoryDescrip or "",
                federal_share_obligated=r.federalShareObligated or Decimal("0"),
            )
            for r in rows
        ],
    )

In [0]:
# --- Smoke test: call each function and display results --------------------

# Tool 1: statewide PA funding for Texas
result1 = summarize_pa_funding_by_jurisdiction(
    FundingByJurisdictionInput(state=USState(code="TX"))
)
print("Tool 1 — Funding by jurisdiction (TX):")
print(f"  projects: {result1.project_count:,}")
print(f"  federal share obligated: ${result1.total_federal_share_obligated:,.2f}")
print(f"  total obligated:         ${result1.total_obligated:,.2f}")
print()

# Tool 2: damage-category breakdown for a sample disaster
sample_disaster = (
    spark.sql(f"SELECT DISTINCT disasterNumber FROM {_TABLE} ORDER BY disasterNumber DESC LIMIT 1")
    .first()[0]
)
result2 = breakdown_disaster_funding_by_category(
    FundingByDamageCategoryInput(disaster_number=sample_disaster)
)
print(f"Tool 2 — Damage categories for disaster {sample_disaster}:")
for row in result2.rows:
    print(f"  {row.damage_category_code} ({row.damage_category_description}): "
          f"${row.total_federal_share_obligated:,.2f}  ({row.project_count} projects)")
print()

# Tool 3: top 5 largest projects nationwide
result3 = rank_largest_pa_projects(LargestProjectsInput(limit=5))
print("Tool 3 — Top 5 largest projects:")
for row in result3.rows:
    print(f"  PW {row.pw_number} | {row.state} | ${row.federal_share_obligated:,.2f} | {row.application_title[:60]}")

Tool 1 — Funding by jurisdiction (TX):
  projects: 3,716
  federal share obligated: $1,300,521,413.20
  total obligated:         $1,300,521,413.20

Tool 2 — Damage categories for disaster 4944:
  Z (Management Costs): $784,070.37  (1 projects)

Tool 3 — Top 5 largest projects:
  PW 2 | FL | $587,291,218.63 | Expedited Contractual Services Project
  PW 1 | FL | $440,721,848.38 | Expedited Contractual Services Project
  PW 3 | FL | $304,157,672.87 | Expedited Contractual Services Project
  PW 12 | TX | $292,705,166.59 | TDEM City (Statewide) Waterway, ROW, PPDR and CPDR Debris Re
  PW 4341 | NC | $280,013,687.62 | Impacted Area EPM - 100% Cost Share - Activation, Contracts 
